# Week 3 · Task 1: Basic ML Model Implementation

**Algorithm:** Logistic Regression (Classification)

**Dataset:** Titanic (preprocessed, 21 features + `survived` target)

**Goal:** Build an end-to-end ML pipeline — load, split, train, evaluate.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
)

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print('All libraries loaded')

## 2. Load Dataset

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'cleaned_titanic_data.csv')
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}):\n{list(df.columns)}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Target distribution:')
print(df['survived'].value_counts())
print(f"\nSurvival rate: {df['survived'].mean() * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df['survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#ff6b6b', '#51cf66'])
axes[0].set_title('Survival Count', fontweight='bold')
axes[0].set_xticklabels(['Not Survived', 'Survived'], rotation=0)

numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('survived')
corr = df[numeric_cols].corrwith(df['survived']).sort_values(ascending=False).head(10)
sns.barplot(x=corr.values, y=corr.index, ax=axes[1], palette='coolwarm')
axes[1].set_title('Top 10 Correlations with Survived', fontweight='bold')
axes[1].set_xlabel('Correlation')
plt.tight_layout()
plt.savefig('../images/eda_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Selection

We drop only the target column. All 21 features are preprocessed (OHE + engineered).

In [ ]:
TARGET = 'survived'
X = df.drop(columns=[TARGET])
y = df[TARGET]
print(f'Features: {X.shape[1]} columns, {X.shape[0]} samples')
print(f'Target: {y.name}, {y.nunique()} classes')

## 5. Train/Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print(f'\nTrain distribution:\n{y_train.value_counts(normalize=True).mul(100).round(1)}')
print(f'\nTest distribution:\n{y_test.value_counts(normalize=True).mul(100).round(1)}')

## 6. Feature Scaling

In [ ]:
scale_cols = ['age', 'fare', 'sibsp', 'parch', 'family_size']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
cols_found = [c for c in scale_cols if c in X_train.columns]
X_train_scaled[cols_found] = scaler.fit_transform(X_train[cols_found])
X_test_scaled[cols_found] = scaler.transform(X_test[cols_found])
print(f'Scaled: {cols_found}')

## 7. Train Baseline Model — Logistic Regression

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]
print('Model trained successfully')

## 8. Evaluate: Metrics

In [ ]:
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob)
}

print('=== Logistic Regression Performance ===')
for name, val in metrics.items():
    print(f'{name:12s}: {val:.4f}')

In [ ]:
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Not Survived', 'Survived']))

## 9. Evaluate: Cross-Validation

In [ ]:
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='f1')
print(f'5-Fold CV F1 Scores: {cv_scores.round(4)}')
print(f'Mean CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

## 10. Visualize: Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'])
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('../images/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Visualize: ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

ax.plot(fpr, tpr, lw=2, color='steelblue', label=f'Logistic Regression (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.15, color='steelblue')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Logistic Regression', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../images/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Visualize: Feature Coefficients

In [ ]:
coeff_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coeff_df['Coefficient']]
sns.barplot(data=coeff_df, x='Coefficient', y='Feature', palette=colors, ax=ax)
ax.set_title('Top 12 Feature Coefficients — Logistic Regression', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('../images/feature_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Visualize: Metrics Summary Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
names = list(metrics.keys())
values = list(metrics.values())
bars = ax.bar(names, values, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'], edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Metrics', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
plt.tight_layout()
plt.savefig('../images/metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Model Limitations & Improvement Ideas

### Limitations
1. **Class imbalance** — 62% not survived vs 38% survived. F1-Score accounts for this better than accuracy.
2. **Linear decision boundary** — Logistic Regression assumes linear relationships between features and log-odds.
3. **Feature engineering ceiling** — Gains from tuning are limited without new features or interaction terms.
4. **Cabin information lost** — 77% of Cabin values were missing; replaced with 'Unknown'.

### Potential Improvements
1. Try tree-based models (Random Forest, Gradient Boosting) for non-linear patterns.
2. Apply SMOTE or class weighting to address imbalance.
3. Feature engineering: ticket group size, cabin deck, name title grouping.
4. Hyperparameter tuning via GridSearchCV (C, penalty, solver).
5. Ensemble: Voting classifier combining Logistic Regression + Random Forest.